# Argus Vision — Full Evaluation Report

**Adversarial multi-agent visual debate for uncertainty-aware medical image classification.**

This notebook benchmarks the full Argus / VisDebate system against single-agent and ensemble baselines, on both the full test set and the hard subset $D_{hard}$ from notebook 03.

Contents:
1. Load all models (Agent A, Agent B, the argument encoder, the consensus head).
2. Evaluate **five** configurations: Agent A only, Agent B only, standard ensemble (mean of $p_A, p_B$), deep ensemble (if available — gracefully skipped otherwise), and the full Argus system.
3. Metrics table (Balanced Accuracy, Macro AUC, ECE) on the full test set.
4. The same metrics restricted to $D_{hard}$.
5. An ablation table (no-debate vs debate, no-attention-features).
6. Six case studies (2 agree-correct, 2 debate-helps, 1 debate-fails, 1 melanoma) rendering the original image, both heatmaps, the $M_\Delta$ disagreement map, argument texts, and confidence before/after debate.

All maths (trigger, attention, disagreement, feature vector, temperature) mirrors the backend exactly so the offline numbers reflect deployed behaviour.

## 1. Environment, constants and the shared contract

In [ ]:
import os
import subprocess
from pathlib import Path

subprocess.run(
    ["pip", "install", "-q", "-r", "requirements_training.txt"],
    check=True,
)
subprocess.run(
    ["pip", "install", "-q", "scipy==1.13.0", "grad-cam==1.5.0",
     "opencv-python-headless==4.9.0.80", "groq==0.9.0", "netcal==1.3.5"],
    check=True,
)

import json  # noqa: E402
from typing import Any, Dict, List, Optional, Tuple  # noqa: E402

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import torch  # noqa: E402
import torch.nn as nn  # noqa: E402
import torch.nn.functional as F  # noqa: E402

# --- Shared contract: ISIC-8 taxonomy ------------------------------------
CLASS_NAMES: List[str] = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"]
FULL_NAMES: Dict[str, str] = {
    "MEL": "Melanoma",
    "NV": "Melanocytic Nevus",
    "BCC": "Basal Cell Carcinoma",
    "AK": "Actinic Keratosis",
    "BKL": "Benign Keratosis",
    "DF": "Dermatofibroma",
    "VASC": "Vascular Lesion",
    "SCC": "Squamous Cell Carcinoma",
}
CLASS_TO_IDX: Dict[str, int] = {name: i for i, name in enumerate(CLASS_NAMES)}
NUM_CLASSES: int = 8
IMAGE_SIZE: int = 224
IMAGENET_MEAN: List[float] = [0.485, 0.456, 0.406]
IMAGENET_STD: List[float] = [0.229, 0.224, 0.225]
EMBEDDING_DIM: int = 384
FEATURE_DIM: int = 788
TAU_JS: float = 0.25
TAU_ENTROPY: float = 0.8

AGENT_A_MODEL_NAME: str = "efficientnet_b4"
AGENT_B_MODEL_NAME: str = "vit_base_patch16_224.augreg_in21k_ft_in1k"
GROQ_MODEL: str = os.environ.get("GROQ_MODEL", "llama-3.3-70b-versatile")
GROQ_API_KEY: str = os.environ.get("GROQ_API_KEY", "")

CHECKPOINT_DIR = Path("../backend/checkpoints")
AGENT_A_CHECKPOINT = CHECKPOINT_DIR / "agent_a_best.pth"
AGENT_B_CHECKPOINT = CHECKPOINT_DIR / "agent_b_best.pth"
CONSENSUS_CHECKPOINT = CHECKPOINT_DIR / "consensus_best.pth"
AGENT_A_DEEP = CHECKPOINT_DIR / "agent_a_best_seed2.pth"
AGENT_B_DEEP = CHECKPOINT_DIR / "agent_b_best_seed2.pth"
ARTIFACT_DIR = Path("./artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
HARD_SUBSET_CSV = ARTIFACT_DIR / "hard_subset.csv"
GROQ_CACHE_PATH = ARTIFACT_DIR / "groq_argument_cache.json"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"Device: {DEVICE}")

## 2. The consensus MLP (same architecture as notebook 04 / the backend)

In [ ]:
class ConsensusMLP(nn.Module):
    """Calibrated consensus classifier over the 788-dim debate feature vector.

    Identical to ``backend/ml/consensus/classifier.py``: ``788->512->256->8``
    with ``BatchNorm1d`` + ``Dropout(0.3)`` after each hidden layer, plus a
    learnable temperature scalar dividing the logits before softmax.

    Attributes:
        backbone: The ordered feature transform producing raw logits.
        temperature: A single learnable temperature scalar (init 1.0).
    """

    def __init__(
        self,
        input_dim: int = FEATURE_DIM,
        num_classes: int = NUM_CLASSES,
        dropout: float = 0.3,
    ) -> None:
        """Initialise the consensus MLP.

        Args:
            input_dim: Dimensionality of the input feature vector (788).
            num_classes: Number of output classes (8 for ISIC-8).
            dropout: Dropout probability applied after each hidden layer.
        """
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )
        self.temperature = nn.Parameter(torch.ones(1))

    def logits(self, features: torch.Tensor) -> torch.Tensor:
        """Return the raw (un-temperature-scaled) logits.

        Args:
            features: A ``(N, input_dim)`` feature batch.

        Returns:
            A ``(N, num_classes)`` tensor of raw logits.
        """
        return self.backbone(features)

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        """Return temperature-scaled logits ready for a softmax.

        Args:
            features: A ``(N, input_dim)`` feature batch.

        Returns:
            A ``(N, num_classes)`` tensor of logits divided by the temperature.
        """
        return self.logits(features) / self.temperature.clamp_min(1e-3)

## 3. Load the test dataframe, $D_{hard}$, the agents and the consensus head

In [ ]:
import timm
from PIL import Image
from sklearn.model_selection import train_test_split
import torchvision.transforms as T

try:
    from config import AgentAConfig  # type: ignore
    from dataset import load_isic_dataframe  # type: ignore

    CFG = AgentAConfig()
    full_df = load_isic_dataframe(CFG)
except Exception as exc:  # noqa: BLE001
    print(f"dataset.py loader unavailable ({exc}); scanning DATA_DIR.")
    DATA_DIR = Path(os.environ.get("ISIC_DATA_DIR", "./data"))
    gt_candidates = sorted(DATA_DIR.rglob("*GroundTruth*.csv"))
    if not gt_candidates:
        raise FileNotFoundError("No ISIC ground-truth CSV found.")
    raw = pd.read_csv(gt_candidates[0])
    image_col = raw.columns[0]
    onehot = raw[CLASS_NAMES].to_numpy()
    image_root = gt_candidates[0].parent
    records: List[Dict[str, object]] = []
    for i in range(len(raw)):
        name = str(raw.iloc[i][image_col])
        matches = list(image_root.rglob(f"{name}.jpg"))
        if matches:
            records.append(
                {"image_path": str(matches[0]), "label": int(onehot[i].argmax())}
            )
    full_df = pd.DataFrame.from_records(records)

full_df = full_df.reset_index(drop=True)
_, holdout_df = train_test_split(
    full_df, test_size=0.20, stratify=full_df["label"], random_state=SEED
)
_, test_df = train_test_split(
    holdout_df, test_size=0.50, stratify=holdout_df["label"], random_state=SEED
)
test_df = test_df.reset_index(drop=True)
print(f"Test-set size: {len(test_df):,}")

hard_paths: set = set()
if HARD_SUBSET_CSV.exists():
    hard_df = pd.read_csv(HARD_SUBSET_CSV)
    hard_paths = set(hard_df["image_path"].astype(str))
    print(f"Loaded {len(hard_paths):,} hard-subset paths from notebook 03.")
else:
    print("hard_subset.csv not found; D_hard tables will be computed on the fly.")

eval_transform = T.Compose(
    [
        T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)


def load_tensor(image_path: str) -> torch.Tensor:
    """Load an image and return a preprocessed ``(1, 3, 224, 224)`` tensor.

    Args:
        image_path: Absolute path to a dermoscopic image on disk.

    Returns:
        A normalized float tensor with a leading batch dimension.
    """
    with Image.open(image_path) as handle:
        image = handle.convert("RGB")
    return eval_transform(image).unsqueeze(0)


def build_agent(model_name: str, checkpoint: Path,
                required: bool = False) -> Optional[nn.Module]:
    """Construct a timm backbone and load its checkpoint.

    Args:
        model_name: The timm model identifier for the backbone.
        checkpoint: Path to the agent's ``.pth`` state dict.
        required: When ``True`` an ImageNet fallback is used if the checkpoint
            is missing; when ``False`` a missing checkpoint returns ``None``
            (used for the optional deep-ensemble members).

    Returns:
        The model in eval mode on ``DEVICE``, or ``None`` if optional + missing.
    """
    has_ckpt = checkpoint.exists()
    if not has_ckpt and not required:
        return None
    model = timm.create_model(
        model_name, pretrained=not has_ckpt, num_classes=NUM_CLASSES
    )
    if has_ckpt:
        state = torch.load(checkpoint, map_location=DEVICE)
        if isinstance(state, dict) and "state_dict" in state:
            state = state["state_dict"]
        model.load_state_dict(state, strict=False)
        print(f"Loaded checkpoint {checkpoint.name}.")
    else:
        print(f"WARNING: {checkpoint.name} missing; using ImageNet fallback.")
    return model.eval().to(DEVICE)


agent_a = build_agent(AGENT_A_MODEL_NAME, AGENT_A_CHECKPOINT, required=True)
agent_b = build_agent(AGENT_B_MODEL_NAME, AGENT_B_CHECKPOINT, required=True)
agent_a_deep = build_agent(AGENT_A_MODEL_NAME, AGENT_A_DEEP, required=False)
agent_b_deep = build_agent(AGENT_B_MODEL_NAME, AGENT_B_DEEP, required=False)
DEEP_ENSEMBLE_AVAILABLE = agent_a_deep is not None and agent_b_deep is not None
print(f"Deep ensemble available: {DEEP_ENSEMBLE_AVAILABLE}")

consensus: Optional[ConsensusMLP] = None
if CONSENSUS_CHECKPOINT.exists():
    consensus = ConsensusMLP().to(DEVICE)
    consensus.load_state_dict(
        torch.load(CONSENSUS_CHECKPOINT, map_location=DEVICE)
    )
    consensus.eval()
    print(f"Loaded consensus head (T={float(consensus.temperature.item()):.4f}).")
else:
    print("WARNING: consensus_best.pth missing; Argus rows use the ensemble path.")

## 4. Argus building blocks (attention, disagreement, trigger, encoder, debate)

These reproduce the backend modules so the full system can be run end-to-end on every test image. The Groq cache from notebook 04 is reused read-only here — no new paid calls are made when a transcript is already cached.

In [ ]:
import re

import cv2
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from scipy.spatial.distance import jensenshannon
from sentence_transformers import SentenceTransformer

_EPS = 1e-12
GRID_SIZE = 14
NUM_PATCH_TOKENS = GRID_SIZE * GRID_SIZE
_DELTA_PATTERN = re.compile(r"CONFIDENCE_DELTA\s*:\s*([+-]?\d*\.?\d+)", re.IGNORECASE)

encoder = SentenceTransformer("all-MiniLM-L6-v2", device=DEVICE)


def encode_argument(text: str) -> np.ndarray:
    """Encode an argument string into a 384-d L2-normalised embedding."""
    if not text or not text.strip():
        return np.zeros(EMBEDDING_DIM, dtype=np.float32)
    return np.asarray(
        encoder.encode(text, normalize_embeddings=True), dtype=np.float32
    )


def shannon_entropy(probs: np.ndarray) -> float:
    """Compute the base-2 Shannon entropy (bits) of a probability vector."""
    p = np.clip(probs, 0.0, None)
    return -float(np.sum(p * np.log2(p + _EPS)))


def js_divergence(probs_a: np.ndarray, probs_b: np.ndarray) -> float:
    """Compute the squared Jensen-Shannon divergence between two vectors."""
    distance = jensenshannon(probs_a, probs_b, base=2)
    divergence = float(distance) ** 2
    return divergence if np.isfinite(divergence) else 0.0


def trigger_fires(div: float, ent_a: float, ent_b: float) -> bool:
    """Replicate the backend debate-trigger decision."""
    return (div > TAU_JS) or (max(ent_a, ent_b) > TAU_ENTROPY)


def gradcam_plusplus(model: nn.Module, tensor: torch.Tensor, target: int) -> np.ndarray:
    """Compute a 224x224 Grad-CAM++ saliency map for Agent A."""
    blocks = getattr(model, "blocks", None)
    if blocks is not None and len(blocks) > 0:
        last = blocks[-1]
        target_layer = getattr(last, "bn3", last)
    else:
        target_layer = getattr(model, "conv_head", None)
        if target_layer is None:
            for module in model.modules():
                if isinstance(module, (nn.BatchNorm2d, nn.Conv2d)):
                    target_layer = module
    model.zero_grad()
    cam = GradCAMPlusPlus(model=model, target_layers=[target_layer])
    grayscale = cam(input_tensor=tensor, targets=[ClassifierOutputTarget(target)])
    model.zero_grad()
    heatmap = np.asarray(grayscale[0], dtype=np.float32)
    if heatmap.shape != (IMAGE_SIZE, IMAGE_SIZE):
        heatmap = cv2.resize(heatmap, (IMAGE_SIZE, IMAGE_SIZE),
                             interpolation=cv2.INTER_LINEAR)
    return np.clip(heatmap, 0.0, 1.0).astype(np.float32)


def attention_rollout(model: nn.Module, tensor: torch.Tensor) -> np.ndarray:
    """Compute a 224x224 attention-rollout saliency map for Agent B."""
    handles: List[Any] = []
    captured: Dict[int, torch.Tensor] = {}
    original_fused: Dict[int, bool] = {}
    blocks = getattr(model, "blocks", None)

    def make_hook(layer_idx: int):
        def hook(module: nn.Module, inputs: tuple, output: torch.Tensor) -> None:
            x = inputs[0]
            batch, num_tokens, dim = x.shape
            heads = int(module.num_heads)
            head_dim = dim // heads
            scale = getattr(module, "scale", head_dim ** -0.5)
            qkv = module.qkv(x).reshape(batch, num_tokens, 3, heads, head_dim)
            qkv = qkv.permute(2, 0, 3, 1, 4)
            q, k = qkv[0], qkv[1]
            attn = (q @ k.transpose(-2, -1)) * float(scale)
            attn = attn.softmax(dim=-1)
            captured[layer_idx] = attn.mean(dim=1).detach().to(torch.float32).cpu()
        return hook

    try:
        for idx, block in enumerate(blocks):
            attn_module = getattr(block, "attn", None)
            if attn_module is None or not hasattr(attn_module, "qkv"):
                continue
            original_fused[idx] = bool(getattr(attn_module, "fused_attn", False))
            if hasattr(attn_module, "fused_attn"):
                attn_module.fused_attn = False
            handles.append(attn_module.register_forward_hook(make_hook(idx)))
        with torch.no_grad():
            model(tensor)
        layer_indices = sorted(captured.keys())
        num_tokens = captured[layer_indices[0]].shape[-1]
        identity = torch.eye(num_tokens, dtype=torch.float32)
        rollout = torch.eye(num_tokens, dtype=torch.float32)
        for idx in layer_indices:
            attn = captured[idx][0]
            aug = 0.5 * attn + 0.5 * identity
            aug = aug / aug.sum(dim=-1, keepdim=True).clamp_min(1e-12)
            rollout = aug @ rollout
        cls_attention = rollout[0, 1:1 + NUM_PATCH_TOKENS]
        grid = cls_attention.reshape(GRID_SIZE, GRID_SIZE).numpy().astype(np.float32)
        heatmap = cv2.resize(grid, (IMAGE_SIZE, IMAGE_SIZE),
                             interpolation=cv2.INTER_CUBIC).astype(np.float32)
        span = float(heatmap.max() - heatmap.min())
        if span < 1e-12:
            return np.zeros((IMAGE_SIZE, IMAGE_SIZE), dtype=np.float32)
        return ((heatmap - heatmap.min()) / span).astype(np.float32)
    finally:
        for handle in handles:
            handle.remove()
        for idx, was_fused in original_fused.items():
            attn_module = getattr(blocks[idx], "attn", None)
            if attn_module is not None and hasattr(attn_module, "fused_attn"):
                attn_module.fused_attn = was_fused


def disagreement_map(heatmap_a: np.ndarray, heatmap_b: np.ndarray) -> np.ndarray:
    """Return the absolute-difference disagreement map ``M_delta`` in ``[0, 1]``."""
    def normalize(arr: np.ndarray) -> np.ndarray:
        arr = np.asarray(arr, dtype=np.float32)
        span = float(arr.max() - arr.min())
        return np.zeros_like(arr) if span < 1e-12 else (arr - arr.min()) / span
    return np.abs(normalize(heatmap_a) - normalize(heatmap_b)).astype(np.float32)


def region_statistics(
    heatmap_a: np.ndarray, heatmap_b: np.ndarray
) -> Tuple[Dict[str, float], Dict[str, float]]:
    """Compute contested-region (top-20%-mass) stats for both agents."""
    def normalize(arr: np.ndarray) -> np.ndarray:
        arr = np.asarray(arr, dtype=np.float32)
        span = float(arr.max() - arr.min())
        return np.zeros_like(arr) if span < 1e-12 else (arr - arr.min()) / span

    def stats(norm_map: np.ndarray, mask: np.ndarray) -> Dict[str, float]:
        if not bool(mask.any()):
            return {"mean": 0.0, "std": 0.0, "max": 0.0}
        sel = norm_map[mask]
        return {"mean": float(sel.mean()), "std": float(sel.std()),
                "max": float(sel.max())}

    norm_a, norm_b = normalize(heatmap_a), normalize(heatmap_b)
    combined = norm_a + norm_b
    mask = combined >= float(np.percentile(combined, 80.0))
    if not bool(mask.any()):
        mask = np.ones_like(combined, dtype=bool)
    return stats(norm_a, mask), stats(norm_b, mask)


# Reuse the read-only Groq cache produced by notebook 04.
if GROQ_CACHE_PATH.exists():
    with open(GROQ_CACHE_PATH, "r", encoding="utf-8") as handle:
        GROQ_CACHE: Dict[str, Dict[str, Any]] = json.load(handle)
    print(f"Loaded {len(GROQ_CACHE)} cached debate transcripts.")
else:
    GROQ_CACHE = {}
    print("No Groq cache found; debate arguments use deterministic fallbacks.")


def _fallback_argument(pred: str, conf: float, stats: Dict[str, float]) -> str:
    """Build a deterministic offline argument string."""
    return (
        f"This lesion is most consistent with {FULL_NAMES[pred]} ({pred}) at "
        f"{conf * 100:.1f}% confidence, with contested-region mean activation "
        f"{stats.get('mean', 0.0):.3f} and peak {stats.get('max', 0.0):.3f}."
    )


def get_arguments(
    image_id: str, pred_a: str, conf_a: float, stats_a: Dict[str, float],
    pred_b: str, conf_b: float, stats_b: Dict[str, float],
) -> Tuple[str, str]:
    """Return the round-2 arguments (A, B), preferring the cached transcript.

    Args:
        image_id: The per-image cache key (filename stem).
        pred_a: Agent A's predicted class code.
        conf_a: Agent A's confidence (0-1).
        stats_a: Agent A contested-region stats.
        pred_b: Agent B's predicted class code.
        conf_b: Agent B's confidence (0-1).
        stats_b: Agent B contested-region stats.

    Returns:
        A ``(argument_a, argument_b)`` tuple of round-2 argument strings with
        any trailing ``CONFIDENCE_DELTA`` line removed.
    """
    if image_id in GROQ_CACHE:
        transcript = GROQ_CACHE[image_id]
        arg_a = _DELTA_PATTERN.sub("", transcript.get("arg_a_r2", "")).strip()
        arg_b = _DELTA_PATTERN.sub("", transcript.get("arg_b_r2", "")).strip()
        return arg_a, arg_b
    return (
        _fallback_argument(pred_a, conf_a, stats_a),
        _fallback_argument(pred_b, conf_b, stats_b),
    )


print("Argus building blocks ready.")

## 5. Run all five configurations over the full test set

For each test image we record the probability vector produced by every configuration. On the **fast path** (trigger does not fire) the Argus feature vector uses zeros for `spatial_stats` and zero-vectors for `eA`/`eB`, exactly as the contract specifies — so the consensus head still produces a calibrated distribution. On the **debate path** the full attention + argument features are populated.

We also store the ablation variants needed for section 8 in the same single pass to avoid recomputing attention twice.

In [ ]:
from tqdm.auto import tqdm


@torch.no_grad()
def predict_probs(model: nn.Module, tensor: torch.Tensor) -> np.ndarray:
    """Return the softmax probability vector for a single image tensor."""
    logits = model(tensor.to(DEVICE))
    return F.softmax(logits, dim=1)[0].detach().cpu().numpy().astype(np.float64)


def assemble_feature(
    pa: np.ndarray, pb: np.ndarray, spatial: np.ndarray,
    emb_a: np.ndarray, emb_b: np.ndarray,
) -> np.ndarray:
    """Concatenate the 788-dim consensus feature vector.

    Args:
        pa: Agent A probabilities ``(8,)``.
        pb: Agent B probabilities ``(8,)``.
        spatial: The 4-dim ``[mean_a, mean_b, std_a, std_b]`` stats.
        emb_a: Agent A argument embedding ``(384,)``.
        emb_b: Agent B argument embedding ``(384,)``.

    Returns:
        A ``float32`` feature vector of shape ``(788,)``.
    """
    return np.concatenate(
        [pa.astype(np.float32), pb.astype(np.float32), spatial.astype(np.float32),
         emb_a.astype(np.float32), emb_b.astype(np.float32)]
    )


def consensus_probs(feature: np.ndarray) -> np.ndarray:
    """Run the consensus head on a single 788-dim feature (eval-safe for BN)."""
    if consensus is None:
        # No consensus head: fall back to the standard ensemble distribution.
        return (feature[:NUM_CLASSES] + feature[NUM_CLASSES:2 * NUM_CLASSES]) / 2.0
    batch = torch.from_numpy(feature).float().unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = consensus(batch)
    return F.softmax(logits, dim=1)[0].cpu().numpy().astype(np.float64)


y_true: List[int] = []
is_hard: List[bool] = []
probs_store: Dict[str, List[np.ndarray]] = {
    "Agent A": [], "Agent B": [], "Ensemble": [], "Deep Ensemble": [],
    "Argus": [], "Argus (no debate)": [], "Argus (no attention)": [],
}
records: List[Dict[str, Any]] = []
ZERO_SPATIAL = np.zeros(4, dtype=np.float32)
ZERO_EMB = np.zeros(EMBEDDING_DIM, dtype=np.float32)

for record in tqdm(test_df.itertuples(index=False), total=len(test_df)):
    image_path = str(record.image_path)
    image_id = Path(image_path).stem
    try:
        tensor = load_tensor(image_path)
    except Exception as exc:  # noqa: BLE001
        print(f"Skipping unreadable {image_path}: {exc}")
        continue
    true_idx = int(record.label)

    pa = predict_probs(agent_a, tensor)
    pb = predict_probs(agent_b, tensor)
    ensemble = (pa + pb) / 2.0

    fired = trigger_fires(js_divergence(pa, pb), shannon_entropy(pa),
                          shannon_entropy(pb))

    # Debate-path features (only computed when the trigger fires).
    spatial = ZERO_SPATIAL.copy()
    emb_a, emb_b = ZERO_EMB.copy(), ZERO_EMB.copy()
    heatmap_a = heatmap_b = None
    stats_a: Dict[str, float] = {"mean": 0.0, "std": 0.0, "max": 0.0}
    stats_b: Dict[str, float] = {"mean": 0.0, "std": 0.0, "max": 0.0}
    arg_a = arg_b = ""
    if fired:
        heatmap_a = gradcam_plusplus(agent_a, tensor, int(pa.argmax()))
        heatmap_b = attention_rollout(agent_b, tensor)
        stats_a, stats_b = region_statistics(heatmap_a, heatmap_b)
        spatial = np.array(
            [stats_a["mean"], stats_b["mean"], stats_a["std"], stats_b["std"]],
            dtype=np.float32,
        )
        arg_a, arg_b = get_arguments(
            image_id, CLASS_NAMES[int(pa.argmax())], float(pa.max()), stats_a,
            CLASS_NAMES[int(pb.argmax())], float(pb.max()), stats_b,
        )
        emb_a = encode_argument(arg_a)
        emb_b = encode_argument(arg_b)

    # Full Argus feature + the two ablation features.
    feat_full = assemble_feature(pa, pb, spatial, emb_a, emb_b)
    feat_no_debate = assemble_feature(pa, pb, ZERO_SPATIAL, ZERO_EMB, ZERO_EMB)
    feat_no_attention = assemble_feature(pa, pb, ZERO_SPATIAL, emb_a, emb_b)

    argus = consensus_probs(feat_full)
    argus_no_debate = consensus_probs(feat_no_debate)
    argus_no_attention = consensus_probs(feat_no_attention)

    y_true.append(true_idx)
    is_hard.append(image_path in hard_paths if hard_paths else fired)
    probs_store["Agent A"].append(pa)
    probs_store["Agent B"].append(pb)
    probs_store["Ensemble"].append(ensemble)
    probs_store["Argus"].append(argus)
    probs_store["Argus (no debate)"].append(argus_no_debate)
    probs_store["Argus (no attention)"].append(argus_no_attention)
    if DEEP_ENSEMBLE_AVAILABLE:
        pa2 = predict_probs(agent_a_deep, tensor)
        pb2 = predict_probs(agent_b_deep, tensor)
        probs_store["Deep Ensemble"].append((pa + pb + pa2 + pb2) / 4.0)

    records.append(
        {
            "image_path": image_path, "image_id": image_id, "true_idx": true_idx,
            "fired": fired, "pa": pa, "pb": pb, "argus": argus,
            "ensemble": ensemble, "arg_a": arg_a, "arg_b": arg_b,
            "heatmap_a": heatmap_a, "heatmap_b": heatmap_b,
        }
    )

y_true_arr = np.asarray(y_true, dtype=np.int64)
is_hard_arr = np.asarray(is_hard, dtype=bool)
print(f"Evaluated {len(y_true_arr):,} images; "
      f"{int(is_hard_arr.sum()):,} in D_hard.")

## 6. Metric helpers and the full-test-set table

We report **Balanced Accuracy**, **Macro AUC** (one-vs-rest, averaged over classes present in the slice) and **ECE** (`netcal`, 15 bins) for every configuration.

In [ ]:
from netcal.metrics import ECE
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

_ece_metric = ECE(bins=15)


def compute_metrics(
    probs: np.ndarray, labels: np.ndarray
) -> Dict[str, float]:
    """Compute balanced accuracy, macro AUC and ECE for one configuration.

    Args:
        probs: A ``(N, 8)`` array of predicted probability vectors.
        labels: A ``(N,)`` array of integer ground-truth class indices.

    Returns:
        A dict with ``balanced_accuracy``, ``macro_auc`` and ``ece`` floats.
        Macro AUC is ``nan`` when fewer than two classes are present.
    """
    preds = probs.argmax(axis=1)
    bal_acc = float(balanced_accuracy_score(labels, preds))
    present = np.unique(labels)
    if present.size < 2:
        macro_auc = float("nan")
    else:
        try:
            macro_auc = float(
                roc_auc_score(
                    labels, probs, multi_class="ovr", average="macro",
                    labels=list(range(NUM_CLASSES)),
                )
            )
        except ValueError:
            # Restrict to present classes if some are absent in this slice.
            macro_auc = float(
                roc_auc_score(
                    labels, probs[:, present], multi_class="ovr",
                    average="macro", labels=list(present),
                )
            )
    ece = float(_ece_metric.measure(probs, labels))
    return {"balanced_accuracy": bal_acc, "macro_auc": macro_auc, "ece": ece}


def metrics_table(
    config_names: List[str], mask: Optional[np.ndarray] = None
) -> pd.DataFrame:
    """Build a metrics dataframe over the named configurations.

    Args:
        config_names: Configuration keys to include (must have stored probs).
        mask: Optional boolean row mask selecting a slice (e.g. ``is_hard``).

    Returns:
        A dataframe indexed by configuration with the three metric columns.
    """
    rows: Dict[str, Dict[str, float]] = {}
    labels = y_true_arr if mask is None else y_true_arr[mask]
    for name in config_names:
        if not probs_store[name]:
            continue
        stacked = np.vstack(probs_store[name])
        sliced = stacked if mask is None else stacked[mask]
        if sliced.shape[0] == 0:
            continue
        rows[name] = compute_metrics(sliced, labels)
    return pd.DataFrame.from_dict(rows, orient="index")


main_configs = ["Agent A", "Agent B", "Ensemble"]
if DEEP_ENSEMBLE_AVAILABLE:
    main_configs.append("Deep Ensemble")
main_configs.append("Argus")

full_table = metrics_table(main_configs)
full_table.to_csv(ARTIFACT_DIR / "metrics_full_test.csv")
print("=== Full test set ===")
full_table.round(4)

## 7. Metrics restricted to the hard subset $D_{hard}$

This is where the debate machinery is supposed to earn its keep: on the contested cases the single agents and the naive ensemble degrade, and Argus should recover accuracy and (especially) calibration.

In [ ]:
hard_table = metrics_table(main_configs, mask=is_hard_arr)
hard_table.to_csv(ARTIFACT_DIR / "metrics_hard_subset.csv")
print(f"=== Hard subset D_hard ({int(is_hard_arr.sum()):,} images) ===")
hard_table.round(4)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, metric, title in zip(
    axes,
    ["balanced_accuracy", "macro_auc", "ece"],
    ["Balanced Accuracy", "Macro AUC", "ECE (lower is better)"],
):
    width = 0.4
    idx = np.arange(len(full_table.index))
    ax.bar(idx - width / 2, full_table[metric].values, width,
           label="Full test", color="#3B7DD8")
    hard_vals = hard_table.reindex(full_table.index)[metric].values
    ax.bar(idx + width / 2, hard_vals, width, label="$D_{hard}$", color="#D4A017")
    ax.set_xticks(idx)
    ax.set_xticklabels(full_table.index, rotation=30, ha="right")
    ax.set_title(title)
    ax.legend()
fig.suptitle("Configuration metrics: full test vs hard subset", y=1.02)
fig.tight_layout()
fig.savefig(ARTIFACT_DIR / "metrics_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Ablation table

We isolate the contribution of the debate features by comparing three consensus variants on $D_{hard}$:
* **Argus (full)** — `[pA, pB, spatial_stats, eA, eB]`.
* **Argus (no debate)** — `[pA, pB, 0, 0, 0]` (probabilities only; the fast-path feature for every image).
* **Argus (no attention)** — `[pA, pB, 0, eA, eB]` (argument embeddings kept, spatial stats zeroed).

Comparing against the naive **Ensemble** baseline shows how much of the gain comes from the learned head vs the debate/attention signal specifically.

In [ ]:
ablation_configs = [
    "Ensemble", "Argus (no debate)", "Argus (no attention)", "Argus",
]
ablation_table = metrics_table(ablation_configs, mask=is_hard_arr)
ablation_table = ablation_table.rename(index={"Argus": "Argus (full)"})
ablation_table.to_csv(ARTIFACT_DIR / "metrics_ablation.csv")
print("=== Ablation on D_hard ===")
ablation_table.round(4)

## 9. Six case studies

We select six representative cases and render, for each: the original image, Agent A's Grad-CAM++, Agent B's attention rollout, the $M_\Delta$ disagreement map, the two round-2 argument texts, and the confidence in the true class **before** debate (the naive ensemble) vs **after** debate (the Argus consensus).

Categories:
1. & 2. **agree-correct** — both agents agree and are right.
3. & 4. **debate-helps** — the ensemble is wrong but Argus recovers the truth.
5. **debate-fails** — a contested case Argus still gets wrong (honest reporting).
6. **melanoma** — a high-risk MEL case, the headline clinical scenario.

In [ ]:
def ensemble_correct(rec: Dict[str, Any]) -> bool:
    """Whether the naive ensemble predicts the true class for a record."""
    return int(rec["ensemble"].argmax()) == rec["true_idx"]


def argus_correct(rec: Dict[str, Any]) -> bool:
    """Whether the Argus consensus predicts the true class for a record."""
    return int(rec["argus"].argmax()) == rec["true_idx"]


def agents_agree(rec: Dict[str, Any]) -> bool:
    """Whether the two agents' argmax predictions coincide."""
    return int(rec["pa"].argmax()) == int(rec["pb"].argmax())


def pick(predicate, taken: set, limit: int) -> List[Dict[str, Any]]:
    """Select up to ``limit`` records satisfying ``predicate`` and not yet used.

    Args:
        predicate: A callable mapping a record to ``bool``.
        taken: A set of already-selected ``image_id`` values (mutated in place).
        limit: The maximum number of records to return.

    Returns:
        A list of selected record dicts.
    """
    chosen: List[Dict[str, Any]] = []
    for rec in records:
        if len(chosen) >= limit:
            break
        if rec["image_id"] in taken:
            continue
        if predicate(rec):
            chosen.append(rec)
            taken.add(rec["image_id"])
    return chosen


taken_ids: set = set()
agree_correct = pick(
    lambda r: agents_agree(r) and ensemble_correct(r) and argus_correct(r),
    taken_ids, 2,
)
debate_helps = pick(
    lambda r: r["fired"] and not ensemble_correct(r) and argus_correct(r),
    taken_ids, 2,
)
debate_fails = pick(
    lambda r: r["fired"] and not argus_correct(r),
    taken_ids, 1,
)
melanoma = pick(
    lambda r: r["true_idx"] == CLASS_TO_IDX["MEL"],
    taken_ids, 1,
)

case_studies: List[Tuple[str, Dict[str, Any]]] = (
    [("agree-correct", r) for r in agree_correct]
    + [("debate-helps", r) for r in debate_helps]
    + [("debate-fails", r) for r in debate_fails]
    + [("melanoma", r) for r in melanoma]
)
print(f"Selected {len(case_studies)} case studies:")
for label, rec in case_studies:
    print(f"  [{label}] {rec['image_id']} (GT={CLASS_NAMES[rec['true_idx']]})")

In [ ]:
def denormalize(tensor: torch.Tensor) -> np.ndarray:
    """Undo ImageNet normalisation and return an ``(H, W, 3)`` uint8 image."""
    mean = np.array(IMAGENET_MEAN).reshape(3, 1, 1)
    std = np.array(IMAGENET_STD).reshape(3, 1, 1)
    arr = tensor[0].cpu().numpy() * std + mean
    arr = np.clip(arr.transpose(1, 2, 0), 0.0, 1.0)
    return (arr * 255).astype(np.uint8)


def overlay(base: np.ndarray, heatmap: Optional[np.ndarray]) -> np.ndarray:
    """Blend a ``[0, 1]`` heatmap over an RGB image as a JET overlay."""
    if heatmap is None:
        return base
    coloured = cv2.applyColorMap(
        (np.clip(heatmap, 0.0, 1.0) * 255).astype(np.uint8), cv2.COLORMAP_JET
    )
    coloured = cv2.cvtColor(coloured, cv2.COLOR_BGR2RGB)
    return (0.55 * base + 0.45 * coloured).astype(np.uint8)


for label, rec in case_studies:
    tensor = load_tensor(rec["image_path"])
    base = denormalize(tensor)
    heatmap_a = rec["heatmap_a"]
    heatmap_b = rec["heatmap_b"]
    if heatmap_a is None:
        heatmap_a = gradcam_plusplus(agent_a, tensor, int(rec["pa"].argmax()))
    if heatmap_b is None:
        heatmap_b = attention_rollout(agent_b, tensor)
    m_delta = disagreement_map(heatmap_a, heatmap_b)

    true_code = CLASS_NAMES[rec["true_idx"]]
    conf_before = float(rec["ensemble"][rec["true_idx"]])
    conf_after = float(rec["argus"][rec["true_idx"]])
    argus_pred = CLASS_NAMES[int(rec["argus"].argmax())]
    ens_pred = CLASS_NAMES[int(rec["ensemble"].argmax())]

    fig, axes = plt.subplots(1, 4, figsize=(17, 4.4))
    axes[0].imshow(base)
    axes[0].set_title(f"Original | GT={true_code}")
    axes[1].imshow(overlay(base, heatmap_a))
    axes[1].set_title(f"Agent A Grad-CAM++ ({CLASS_NAMES[int(rec['pa'].argmax())]})")
    axes[2].imshow(overlay(base, heatmap_b))
    axes[2].set_title(f"Agent B rollout ({CLASS_NAMES[int(rec['pb'].argmax())]})")
    axes[3].imshow(m_delta, cmap="magma")
    axes[3].set_title("$M_\\Delta$ disagreement")
    for ax in axes:
        ax.axis("off")
    correct_colour = "#22C55E" if argus_pred == true_code else "#EF4444"
    fig.suptitle(
        f"[{label}] {rec['image_id']}  |  Ensemble->{ens_pred}  "
        f"Argus->{argus_pred}  |  P(true) {conf_before:.3f} -> {conf_after:.3f}",
        y=1.04, color=correct_colour, fontsize=12,
    )
    fig.tight_layout()
    fig.savefig(ARTIFACT_DIR / f"case_{label}_{rec['image_id']}.png", dpi=150,
                bbox_inches="tight")
    plt.show()

    arg_a = rec["arg_a"] or "(fast path: no debate was triggered for this case)"
    arg_b = rec["arg_b"] or "(fast path: no debate was triggered for this case)"
    print(f"--- [{label}] {rec['image_id']} ---")
    print(f"Agent A argument: {arg_a}")
    print(f"Agent B argument: {arg_b}")
    print(f"Confidence in true class {true_code}: "
          f"before(ensemble)={conf_before:.3f} after(Argus)={conf_after:.3f}\n")

## 10. Report summary

All metric CSVs and case-study figures are written to `./artifacts`. The headline comparison is the delta between the naive ensemble and Argus on $D_{hard}$ — both in balanced accuracy/AUC and, critically for clinical deployment, in calibration (ECE).

In [ ]:
def delta_row(table: pd.DataFrame) -> Dict[str, float]:
    """Compute Argus-minus-Ensemble metric deltas for a metrics table."""
    if "Argus" not in table.index or "Ensemble" not in table.index:
        return {}
    return {
        "d_balanced_accuracy": float(
            table.loc["Argus", "balanced_accuracy"]
            - table.loc["Ensemble", "balanced_accuracy"]
        ),
        "d_macro_auc": float(
            table.loc["Argus", "macro_auc"] - table.loc["Ensemble", "macro_auc"]
        ),
        "d_ece": float(table.loc["Argus", "ece"] - table.loc["Ensemble", "ece"]),
    }


print("Argus - Ensemble (full test):", delta_row(full_table))
print("Argus - Ensemble (D_hard):  ", delta_row(hard_table))
print(f"Artifacts written to: {ARTIFACT_DIR.resolve()}")